In [27]:
# Шаг 1: Импорт всего и вся
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('processed_airline_satisfaction.csv')

# Просмотр информации о данных
print("Размер данных:", df.shape)
print("\nПервые 5 строк:")
print(df.head())
print("\nИнформация о данных:")
print(df.info())

Размер данных: (75365, 32)

Первые 5 строк:
   Gender      Customer Type  Age   Type of Travel     Class  Flight Distance  \
0    Male  Disloyal Customer   25  Business travel  Business              235   
1  Female     Loyal Customer   26  Business travel  Business             1142   
2  Female     Loyal Customer   25  Business travel  Business              562   
3    Male     Loyal Customer   61  Business travel  Business              214   
4  Female     Loyal Customer   26  Personal Travel       Eco             1180   

   Inflight wifi service  Departure/Arrival time convenient  \
0                      3                                  2   
1                      2                                  2   
2                      2                                  5   
3                      3                                  3   
4                      3                                  4   

   Ease of Online booking  Gate location  ...             satisfaction  \
0               

In [28]:
# Шаг 2: Разделение на X и y
columns_to_drop = ['satisfaction', 'Date', 'Gender', 'Customer Type', 'Type of Travel', 'Class', 'Age Group']
X = df.drop(columns=columns_to_drop + ['satisfaction_encoded'])
y = df['satisfaction_encoded']

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("\nПризнаки в X:")
print(X.columns.tolist())

Размер X: (75365, 24)
Размер y: (75365,)

Признаки в X:
['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Total Delay', 'Gender_encoded', 'Customer Type_encoded', 'Type of Travel_encoded', 'Class_encoded', 'Age Group_encoded']


In [29]:
# Шаг 3: Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)
print("\nРаспределение классов в y_train:")
print(y_train.value_counts(normalize=True))
print("\nРаспределение классов в y_test:")
print(y_test.value_counts(normalize=True))

Размер обучающей выборки: (52755, 24)
Размер тестовой выборки: (22610, 24)

Распределение классов в y_train:
satisfaction_encoded
0    0.548062
1    0.451938
Name: proportion, dtype: float64

Распределение классов в y_test:
satisfaction_encoded
0    0.548032
1    0.451968
Name: proportion, dtype: float64


In [30]:
# Шаг 4: Масштабирование признаков для логистической регрессии и KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Преобразуем обратно в DataFrame для удобства
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

In [32]:
# Шаг 5: Обучение логистической регрессии с подбором гиперпараметров
print("ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ")

# Определяем параметры для GridSearchCV
param_grid_lr = {
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'max_iter': [100, 200, 500]
}

# Создаем модель
lr = LogisticRegression(random_state=42)

# GridSearchCV с кросс-валидацией
grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Обучение на всей выборке
grid_lr.fit(X_train_scaled_df, y_train)

print(f"Лучшие параметры: {grid_lr.best_params_}")
print(f"Лучшая оценка f1 на кросс-валидации: {grid_lr.best_score_:.4f}")

# Обучение лучшей модели на тренировочных данных
best_lr = grid_lr.best_estimator_
best_lr.fit(X_train_scaled_df, y_train)

# Предсказание на тестовых данных
y_pred_lr = best_lr.predict(X_test_scaled_df)

# Контроль качества
print("\nОценка на тестовой выборке:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_lr):.4f}")

ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
Fitting 5 folds for each of 360 candidates, totalling 1800 fits
Лучшие параметры: {'C': 0.1, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Лучшая оценка f1 на кросс-валидации: 0.8529

Оценка на тестовой выборке:
Accuracy: 0.8686
Precision: 0.8712
Recall: 0.8324
F1-score: 0.8514


In [33]:
# Шаг 6: Обучение KNN с подбором гиперпараметров
print("K-БЛИЖАЙШИХ СОСЕДЕЙ (KNN)")

# Определяем параметры для GridSearchCV
param_grid_knn = {
    'n_neighbors': list(range(3, 11)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Создаем модель
knn = KNeighborsClassifier()

# GridSearchCV с кросс-валидацией
grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Обучение на всей выборке
grid_knn.fit(X_train_scaled_df, y_train)

print(f"Лучшие параметры: {grid_knn.best_params_}")
print(f"Лучшая оценка f1 на кросс-валидации: {grid_knn.best_score_:.4f}")

# Обучение лучшей модели на тренировочных данных
best_knn = grid_knn.best_estimator_
best_knn.fit(X_train_scaled_df, y_train)

# Предсказание на тестовых данных
y_pred_knn = best_knn.predict(X_test_scaled_df)

# Контроль качества
print("\nОценка на тестовой выборке:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_knn):.4f}")
# работает капец как долго

K-БЛИЖАЙШИХ СОСЕДЕЙ (KNN)
ERROR! Session/line number was not unique in database. History logging moved to new session 31
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 10, 'weights': 'distance'}
Лучшая оценка f1 на кросс-валидации: 0.9155

Оценка на тестовой выборке:
Accuracy: 0.9253
Precision: 0.9421
Recall: 0.8892
F1-score: 0.9149


In [ ]:
# Шаг 7: Обучение дерева решений с подбором гиперпараметров
print("ДЕРЕВО РЕШЕНИЙ")

# Определяем параметры для GridSearchCV
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 6]
}

# Создаем модель
dt = DecisionTreeClassifier(random_state=42)

# GridSearchCV с кросс-валидацией
grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Обучение на всей выборке (без масштабирования для дерева решений)
grid_dt.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_dt.best_params_}")
print(f"Лучшая оценка f1 на кросс-валидации: {grid_dt.best_score_:.4f}")

# Обучение лучшей модели на тренировочных данных
best_dt = grid_dt.best_estimator_
best_dt.fit(X_train, y_train)

# Предсказание на тестовых данных
y_pred_dt = best_dt.predict(X_test)

# Контроль качества
print("\nОценка на тестовой выборке:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_dt):.4f}")

In [18]:
# Шаг 8: Анализ важности признаков для дерева решений
print("ВАЖНОСТЬ ПРИЗНАКОВ (ДЕРЕВО РЕШЕНИЙ)")

# Создаем DataFrame с важностью признаков
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_dt.feature_importances_
}).sort_values('importance', ascending=False)

print("Топ-10 наиболее важных признаков:")
print(feature_importance.head(10))

# Проверяем, есть ли среди важных признаков созданные нами (Total Delay, Age Group_encoded)
created_features = ['Total Delay', 'Age Group_encoded']

print("\nВажность созданных признаков:")
for feature in created_features:
    if feature in feature_importance['feature'].values:
        imp = feature_importance[feature_importance['feature'] == feature]['importance'].values[0]
        print(f"{feature}: {imp:.4f}")
    else:
        print(f"{feature}: не найден в признаках")

ВАЖНОСТЬ ПРИЗНАКОВ (ДЕРЕВО РЕШЕНИЙ)
Топ-10 наиболее важных признаков:
                   feature  importance
7          Online boarding    0.410971
2    Inflight wifi service    0.216226
21  Type of Travel_encoded    0.142723
9   Inflight entertainment    0.040971
20   Customer Type_encoded    0.035018
13         Checkin service    0.029295
22           Class_encoded    0.024098
8             Seat comfort    0.013432
5            Gate location    0.013294
14        Inflight service    0.013039

Важность созданных признаков:
Total Delay: 0.0014
Age Group_encoded: 0.0000


In [10]:
# Шаг 9: Применение SMOTE для балансировки данных
print("БАЛАНСИРОВКА ДАННЫХ С ПОМОЩЬЮ SMOTE")

# Применяем SMOTE только к обучающим данным
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled_df, y_train)

print(f"Размер X_train до SMOTE: {X_train_scaled_df.shape}")
print(f"Размер X_train после SMOTE: {X_train_smote.shape}")
print(f"\nРаспределение классов до SMOTE:\n{y_train.value_counts()}")
print(f"\nРаспределение классов после SMOTE:\n{pd.Series(y_train_smote).value_counts()}")

БАЛАНСИРОВКА ДАННЫХ С ПОМОЩЬЮ SMOTE
Размер X_train до SMOTE: (52755, 24)
Размер X_train после SMOTE: (57826, 24)

Распределение классов до SMOTE:
satisfaction_encoded
0    28913
1    23842
Name: count, dtype: int64

Распределение классов после SMOTE:
satisfaction_encoded
1    28913
0    28913
Name: count, dtype: int64


In [11]:
# Шаг 10: Обучение моделей на сбалансированных данных
print("МОДЕЛИ НА СБАЛАНСИРОВАННЫХ ДАННЫХ")

# 1. Логистическая регрессия на сбалансированных данных
print("\n1. Логистическая регрессия (сбалансированные данные):")
best_lr_smote = grid_lr.best_estimator_
best_lr_smote.fit(X_train_smote, y_train_smote)
y_pred_lr_smote = best_lr_smote.predict(X_test_scaled_df)

print(f"Accuracy: {accuracy_score(y_test, y_pred_lr_smote):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr_smote):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr_smote):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_lr_smote):.4f}")

# 2. KNN на сбалансированных данных
print("\n2. KNN (сбалансированные данные):")
best_knn_smote = grid_knn.best_estimator_
best_knn_smote.fit(X_train_smote, y_train_smote)
y_pred_knn_smote = best_knn_smote.predict(X_test_scaled_df)

print(f"Accuracy: {accuracy_score(y_test, y_pred_knn_smote):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn_smote):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_knn_smote):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_knn_smote):.4f}")

# 3. Дерево решений на сбалансированных данных (масштабируем для консистентности)
print("\n3. Дерево решений (сбалансированные данные):")
best_dt_smote = grid_dt.best_estimator_
# Для дерева решений используем немасштабированные данные, преобразованные обратно
X_train_smote_original = pd.DataFrame(
    scaler.inverse_transform(X_train_smote), 
    columns=X.columns
)
best_dt_smote.fit(X_train_smote_original, y_train_smote)
y_pred_dt_smote = best_dt_smote.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_dt_smote):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt_smote):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dt_smote):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_dt_smote):.4f}")

МОДЕЛИ НА СБАЛАНСИРОВАННЫХ ДАННЫХ

1. Логистическая регрессия (сбалансированные данные):
Accuracy: 0.8658
Precision: 0.8531
Recall: 0.8493
F1-score: 0.8512

2. KNN (сбалансированные данные):
Accuracy: 0.9245
Precision: 0.9334
Recall: 0.8971
F1-score: 0.9149

3. Дерево решений (сбалансированные данные):
Accuracy: 0.9479
Precision: 0.9595
Recall: 0.9238
F1-score: 0.9413


In [12]:
# Шаг 11: Сравнение всех моделей
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ ВСЕХ МОДЕЛЕЙ")

# Создаем таблицу для сравнения
results = pd.DataFrame({
    'Модель': [
        'Логистическая регрессия (исходные)',
        'KNN (исходные)',
        'Дерево решений (исходные)',
        'Логистическая регрессия (SMOTE)',
        'KNN (SMOTE)',
        'Дерево решений (SMOTE)'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_lr_smote),
        accuracy_score(y_test, y_pred_knn_smote),
        accuracy_score(y_test, y_pred_dt_smote)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_lr_smote),
        precision_score(y_test, y_pred_knn_smote),
        precision_score(y_test, y_pred_dt_smote)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_lr_smote),
        recall_score(y_test, y_pred_knn_smote),
        recall_score(y_test, y_pred_dt_smote)
    ],
    'F1-score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_lr_smote),
        f1_score(y_test, y_pred_knn_smote),
        f1_score(y_test, y_pred_dt_smote)
    ]
})

print("Таблица сравнения моделей:")
print(results.to_string(index=False))

# Находим лучшую модель по F1-score
best_model_idx = results['F1-score'].idxmax()
best_model = results.loc[best_model_idx]

print(f"\nЛУЧШАЯ МОДЕЛЬ (по F1-score):")
print(f"Модель: {best_model['Модель']}")
print(f"F1-score: {best_model['F1-score']:.4f}")
print(f"Accuracy: {best_model['Accuracy']:.4f}")
print(f"Precision: {best_model['Precision']:.4f}")
print(f"Recall: {best_model['Recall']:.4f}")

СРАВНЕНИЕ РЕЗУЛЬТАТОВ ВСЕХ МОДЕЛЕЙ
Таблица сравнения моделей:
                            Модель  Accuracy  Precision   Recall  F1-score
Логистическая регрессия (исходные)  0.868642   0.871249 0.832371  0.851366
                    KNN (исходные)  0.925254   0.942146 0.889226  0.914921
         Дерево решений (исходные)  0.945997   0.954536 0.924552  0.939305
   Логистическая регрессия (SMOTE)  0.865767   0.853057 0.849300  0.851174
                       KNN (SMOTE)  0.924547   0.933408 0.897055  0.914870
            Дерево решений (SMOTE)  0.947943   0.959545 0.923769  0.941317

ЛУЧШАЯ МОДЕЛЬ (по F1-score):
Модель: Дерево решений (SMOTE)
F1-score: 0.9413
Accuracy: 0.9479
Precision: 0.9595
Recall: 0.9238


In [26]:
# Шаг 12: Заключение
print("ВЫВОДЫ И ЗАКЛЮЧЕНИЕ")

print(f"""
1. КАЧЕСТВО:
   - Все модели показали достаточно хорошее качество - 0.85 :-)
   - F1 варьируется от 0.85 до 0.94

2. ВЛИЯНИЕ БАЛАНСИРОВКИ ДАННЫХ (SMOTE):
   - SMOTE незначительно улучшил метрику recall для некоторых моделей, но в целом не дал существенного прироста в F1-score

3. ВАЖНОСТЬ ПРИЗНАКОВ (для дерева решений):
   - Наиболее важными признаками оказались: Online boarding, Inflight entertainment, Seat comfort, в общем связанные с сервисом
   - Свои признаки окаазались не столь важны :-(, хотя после изначальной обработки казалось что возраст сильно влияет на удовлетворённость,
   возможно если бы я отсекал группы по 15-20 лет эффект был бы больше

4. ЛУЧШАЯ МОДЕЛЬ:
   - Наилучшее качество по F1 показало дерево решений (SMOTE)
   - Лучшие гиперпараметры для дерева решений: {grid_dt.best_params_}
   - F1-score лучшей модели: 0.94

5. РАЗЛИЧИЯ МЕЖДУ МОДЕЛЯМИ:
   - Дерево решений показало наилучшие результаты, вероятно, благодаря способности улавливать нелинейные зависимости в данных
   - Логистическая регрессия и KNN показали сопоставимые результаты, но несколько хуже
   - Различия в качестве между моделями не являются критичными (разница в F1 в пределах 0.06)
""")

ВЫВОДЫ И ЗАКЛЮЧЕНИЕ

1. КАЧЕСТВО:
   - Все модели показали достаточно хорошее качество - 0.85 :-)
   - F1 варьируется от 0.85 до 0.94

2. ВЛИЯНИЕ БАЛАНСИРОВКИ ДАННЫХ (SMOTE):
   - SMOTE незначительно улучшил метрику recall для некоторых моделей, но в целом не дал существенного прироста в F1-score

3. ВАЖНОСТЬ ПРИЗНАКОВ (для дерева решений):
   - Наиболее важными признаками оказались: Online boarding, Inflight entertainment, Seat comfort, в общем связанные с сервисом
   - Свои признаки окаазались не столь важны :-(, хотя после изначальной обработки казалось что возраст сильно влияет на удовлетворённость,
   возможно если бы я отсекал группы по 15-20 лет эффект был бы больше

4. ЛУЧШАЯ МОДЕЛЬ:
   - Наилучшее качество по F1 показало дерево решений (SMOTE)
   - Лучшие гиперпараметры для дерева решений: {'criterion': 'gini', 'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 10}
   - F1-score лучшей модели: 0.94

5. РАЗЛИЧИЯ МЕЖДУ МОДЕЛЯМИ:
   - Дерево решений показало наилучшие р